# LLM-as-a-Judge Analysis

This notebook focuses on the qualitative evaluation of LLM reasoning using an LLM-as-a-judge framework.

## Overview
- Load model reasoning data
- Evaluate dimensions: Factual Accuracy, Context Awareness, Logical Coherence, Comparative Analysis, Depth of Analysis
- Generate summary statistics
- Save detailed results for visualization

In [ ]:
import pandas as pd
import numpy as np
import os
import glob
import json
import sys

# Add current directory to path
sys.path.append(os.getcwd())

from config import *
from utils import load_all_data, save_analysis_results

import asyncio
import nest_asyncio
from openai import AsyncAzureOpenAI
from typing import Dict, List

nest_asyncio.apply()

# Set random seeds
np.random.seed(RANDOM_SEED)

print("✓ All imports successful and config loaded")

In [ ]:
# Configuration imported from config.py
print(f"JUDGE_MODEL: {JUDGE_MODEL}")
print(f"SUBSAMPLE_SIZE: {SUBSAMPLE_SIZE}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")


In [ ]:
class LLMJudge:
    """LLM-based evaluation of reasoning quality"""
    
    def __init__(self, api_key: str, api_version: str, endpoint: str, seed: int = 42):
        self.client = AsyncAzureOpenAI(
            api_key=api_key,
            api_version=api_version,
            azure_endpoint=endpoint
        )
        self.sem = asyncio.Semaphore(CONCURRENCY)
        self.rate_delay = RATE_DELAY
        self.max_retries = MAX_RETRIES
        self.seed = seed
    
    def create_evaluation_prompts(self) -> Dict[str, str]:
        """
        Create separate evaluation prompts for different dimensions.
        Following best practice: use binary classification and chain-of-thought.
        """
        
        prompts = {
            'factual_accuracy': """You are evaluating the factual accuracy of reasoning provided by an LLM for a decision task.

Given the following information:
- Option Selected: {option_selected}
- NPI Level Selected: {npi_selected}
- Infection Rate Selected: {infection_rate_selected}
- NPI Level Non-Selected: {npi_non_selected}
- Infection Rate Non-Selected: {infection_rate_non_selected}
- Context: {context}

Reasoning to evaluate:
{reasoning}

Task: Evaluate whether the reasoning correctly represents the numerical values presented in the options.

First, analyze step-by-step:
1. Does the reasoning correctly identify which option has higher/lower infection rates?
2. Does the reasoning correctly identify which option has more/fewer restrictions?
3. Are the numerical comparisons accurate?

Then provide your evaluation in JSON format:
{{"analysis": "your step-by-step analysis here", "factually_accurate": true/false}}

Factually accurate means ALL numerical facts are correct. If any fact is wrong or misrepresented, mark as false.""",

            'context_awareness': """You are evaluating whether an LLM's reasoning demonstrates awareness of provided context.

Context provided: {context}
Context type: {context_type}

Reasoning to evaluate:
{reasoning}

Task: Determine if the reasoning explicitly considers or mentions the context provided.

First, analyze step-by-step:
1. Is the context variable mentioned or referenced?
2. Does the reasoning explain how context influences the decision?
3. Is the context integrated meaningfully (not just mentioned)?

Then provide your evaluation in JSON format:
{{"analysis": "your step-by-step analysis", "context_aware": true/false}}

Mark as true only if context is meaningfully integrated into the reasoning.""",

            'logical_coherence': """You are evaluating the logical coherence of reasoning.

Reasoning to evaluate:
{reasoning}

Decision made: Selected option with NPI={npi_selected}, Infection={infection_rate_selected}
            Over option with NPI={npi_non_selected}, Infection={infection_rate_non_selected}

Task: Evaluate whether the reasoning follows a logical structure and the conclusion matches the explanation.

First, analyze step-by-step:
1. Does the reasoning present clear criteria for decision-making?
2. Is there a logical flow from analysis to conclusion?
3. Does the stated reasoning actually support the choice made?
4. Are there internal contradictions?

Then provide your evaluation in JSON format:
{{"analysis": "your step-by-step analysis", "logically_coherent": true/false}}

Mark as true only if reasoning is coherent AND supports the decision made.""",

            'comparative_analysis': """You are evaluating whether reasoning includes comparative analysis.

Option A: NPI={npi_selected}, Infection={infection_rate_selected}
Option B: NPI={npi_non_selected}, Infection={infection_rate_non_selected}

Reasoning to evaluate:
{reasoning}

Task: Determine if the reasoning explicitly compares the two options.

First, analyze step-by-step:
1. Are both options mentioned or described?
2. Are trade-offs between options discussed?
3. Are specific differences highlighted?

Then provide your evaluation in JSON format:
{{"analysis": "your step-by-step analysis", "includes_comparison": true/false}}

Mark as true only if there is explicit comparison between the two options.""",

            'depth_of_analysis': """You are evaluating the depth and sophistication of reasoning.

Reasoning to evaluate:
{reasoning}

Task: Determine if the reasoning demonstrates sophisticated analysis beyond surface-level observations.

First, analyze step-by-step:
1. Does it go beyond simply stating the numbers?
2. Does it discuss implications, trade-offs, or broader considerations?
3. Does it show multi-factor thinking?
4. Is there evidence of weighing competing priorities?

Then provide your evaluation in JSON format:
{{"analysis": "your step-by-step analysis", "deep_analysis": true/false}}

Mark as true only if the analysis shows sophisticated, multi-dimensional thinking."""
        }
        
        return prompts
    
    async def evaluate_single_reasoning(
        self, 
        row: pd.Series, 
        prompt_template: str,
        model: str = "gpt-5-mini-2025-08-07",
        temperature: float = 1.0
    ) -> Dict:
        """Evaluate a single reasoning instance"""
        
        # Format the prompt with row data
        context_col = self._get_context_column(row.get('strategy', ''))
        context_value = row.get(context_col, 'None') if context_col else 'None'
        
        prompt = prompt_template.format(
            option_selected=row.get('option_selected', ''),
            npi_selected=row.get('npi_level_selected', ''),
            infection_rate_selected=row.get('infection_rate_selected', ''),
            npi_non_selected=row.get('npi_level_non_selected', ''),
            infection_rate_non_selected=row.get('infection_rate_non_selected', ''),
            context=context_value,
            context_type=context_col,
            reasoning=row.get('reasoning', '')
        )
        
        messages = [{"role": "user", "content": prompt}]
        
        # Retry logic
        backoff = 1.0
        for attempt in range(1, self.max_retries + 1):
            try:
                async with self.sem:
                    if self.rate_delay > 0:
                        await asyncio.sleep(self.rate_delay)
                    
                    response = await self.client.chat.completions.create(
                        model=model,
                        messages=messages,
                        temperature=temperature,
                        seed=self.seed,  # For reproducibility
                        response_format={"type": "json_object"},
                        timeout=30000
                    )
                
                result = json.loads(response.choices[0].message.content)
                return result
                
            except Exception as e:
                if attempt == self.max_retries:
                    print(f"[ERROR] Row index {row.name}: {type(e).__name__}: {e}")
                    return {"analysis": "ERROR", "result": None}
                await asyncio.sleep(backoff + np.random.random() * 0.25 * backoff)
                backoff = min(backoff * 2, 16)
    
    def _get_context_column(self, strategy_name: str) -> str:
        """Extract context column based on strategy"""
        strategy_num = int(''.join(filter(str.isdigit, strategy_name))) if any(c.isdigit() for c in strategy_name) else 0
        context_mapping = {
            2: 'capacity_context',
            3: 'economic_context',
            4: 'user_preference_type'
        }
        return context_mapping.get(strategy_num, None)
    
    async def evaluate_all_dimensions(
        self, 
        df: pd.DataFrame,
        model: str = "gpt-5-mini-2025-08-07",
        temperature: float = 1.0
    ) -> pd.DataFrame:
        """Evaluate all reasoning entries across all dimensions"""
        
        prompts = self.create_evaluation_prompts()
        results = df.copy()
        
        # Mapping from dimension to boolean key in JSON
        result_keys = {
            'factual_accuracy': 'factually_accurate',
            'context_awareness': 'context_aware',
            'logical_coherence': 'logically_coherent',
            'comparative_analysis': 'includes_comparison',
            'depth_of_analysis': 'deep_analysis'
        }
        
        for dimension, prompt_template in prompts.items():
            print(f"\nEvaluating dimension: {dimension}")
            
            tasks = [
                self.evaluate_single_reasoning(row, prompt_template, model, temperature)
                for _, row in df.iterrows()
            ]
            
            dimension_results = await asyncio.gather(*tasks)
            
            # Extract results with explicit key lookup
            expected_key = result_keys.get(dimension)
            
            analyses = []
            scores = []
            
            for r in dimension_results:
                analyses.append(r.get('analysis', 'Error'))
                
                # Try to get the specific score key
                val = r.get(expected_key)
                
                # If not found or null, try to find any boolean
                if val is None:
                    bool_vals = [v for k, v in r.items() if isinstance(v, bool) and k != 'analysis']
                    if bool_vals:
                        val = bool_vals[0]
                
                scores.append(val)
            
            results[f'{dimension}_analysis'] = analyses
            results[f'{dimension}_score'] = scores
        
        return results

In [ ]:
# Load data using utility function
results_df = load_all_data(STRATEGIES, subsample_size=SUBSAMPLE_SIZE, random_seed=RANDOM_SEED)


In [ ]:
# Run Analysis
if not results_df.empty:
    judge = LLMJudge(API_KEY, API_VERSION, RESOURCE_ENDPOINT, seed=RANDOM_SEED)
    analyzed_df = await judge.evaluate_all_dimensions(results_df, JUDGE_MODEL, TEMPERATURE)
    
    # Summarize Results
    print("\nGenerating Summary Statistics...")
    judge_cols = [col for col in analyzed_df.columns if col.endswith('_score')]
    
    agg_dict = {}
    for col in judge_cols:
        agg_dict[col] = lambda x: pd.to_numeric(x, errors='coerce').mean()
    
    if judge_cols:
        summary = analyzed_df.groupby('model').agg(agg_dict).round(3)
        print("\nLLM Judge Summary:")
        display(summary)
    
    # Save Results Granularly
    save_analysis_results(analyzed_df, LLM_JUDGE_DIR, 'llm_judge')
